In [56]:
import pandas as pd
import numpy as np
import nibabel as nib
import nltools 
from bids import BIDSLayout
import timecorr as tc
from nltools.data import Brain_Data
from nltools.utils import get_anatomical

In [57]:
%store -r 

In [58]:
# setup [global vars] 
data_dir = "../../../data-cdl"
layout = BIDSLayout(data_dir)
subs = layout.get_subjects() 

In [82]:
def read_niftis(subs): 
    # nested array of niftis; outer array (index i = sub i-1) & inner arrays (sub i-1 = array of nifti files where index e = run e - 1)
    niftis = [] 
    data_shapes = set()

    # load data 
    for sub_ID in subs: 
        subj_niftis = []
        subj_nifti_filenames = layout.get(subject=sub_ID, return_type='file', task="StudyTest", extension='nii.gz') # array of 8 runs for given sun_ID

        for nifti in subj_nifti_filenames: 
            data = nib.load(nifti)
            subj_niftis.append(data)
            data_shapes.add(data.shape)

        niftis.append(subj_niftis)

    # print(data_shapes)  # {(64, 64, 36, 142)} -- only one shape? 
    return niftis

In [83]:
def check_read_niftis(niftis, subs):
    if len(niftis) != len(subs): 
        print("ERROR: missing some subs")
    for i in range(len(niftis)): 
        sub_ID = f"{int(i)+1:02}"
        if len(niftis[i]) != len(layout.get(subject=sub_ID, return_type='file', task="StudyTest", extension='nii.gz')): 
            print("ERROR: missing some runs for sub ", i+1)

In [84]:
# clip and store timeseries in forget or remember list for each subject 

# each subj is idx in returned timeseries list 
# each idx = [remember list, forget list] (2 element array)
# remember/forget list = list of 4 clipped nifti files 


def clip_niftis(window, subs, timings, cues, niftis): 
    clipped_niftis = [] 
    for sub_idx in range(len(subs)): 
        # sub_idx = 0
        sub_ID = f"{int(sub_idx)+1:02}"

        sub_clipped_niftis = [] 
        sub_niftis = layout.get(subject=sub_ID, return_type='file', task="StudyTest", extension='nii.gz')

        remember = [] # index 0 of sub_timeseries array
        forget = [] # index 1

        for run_idx in range(len(sub_niftis)):
            # find cue timing (TRs)
            cue_tr = timings[sub_idx][run_idx] 
            cue = cues[sub_idx][run_idx]
            
            min_tr = cue_tr - window 
            max_tr = cue_tr + window 

            # get timeseries at window (TRs) around cue 
            nifti = niftis[sub_idx][run_idx]
            data = Brain_Data(nifti)

            clipped_nifti = data[min_tr:max_tr+1]

            if cue == "remember": 
                remember.append(clipped_nifti)
            else: 
                forget.append(clipped_nifti)

        sub_clipped_niftis.append(remember)
        sub_clipped_niftis.append(forget)
        clipped_niftis.append(sub_clipped_niftis)

    return clipped_niftis

In [86]:
def check_clipped_niftis(clipped_niftis): 
    if len(clipped_niftis) != len(subs): 
        print("ERROR: incorrect number of subjects")
    for i in range(len(clipped_niftis)): 
        if len(clipped_niftis[i]) != 2: 
            print("ERROR: supposed to be 2 lists for sub ", i+1, "  (one for each memory condition - forget/remember)")
        if len(clipped_niftis[i][0]) != 4: 
            print("ERROR: missing run(s) in remember list for sub ", i+1)
        if len(clipped_niftis[i][1]) != 4: 
            print("ERROR: missing run(s) in forget list sub ", i+1)        

In [85]:
# determines max window size before error (i.e., clipped nifti file length doesn't match expected window size)

def get_max_window(nifti, cue_tr): 
    for window in range (60): 
        min = cue_tr - window 
        max = cue_tr + window
        
        if len(nifti[min:max+1]) != window*2 + 1: 
            return window

# sub_idx = 0
# run_idx = 0
# nifti = Brain_Data(niftis[sub_idx][run_idx])

# cue_tr = timings[sub_idx][run_idx] 

# max_win = get_max_window(nifti, cue_tr)

# max_win # 46 for sub-01 run 0

c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting data from %s to %s" % (data.dtype.name, aux))


46

In [88]:
def main(): 
    niftis = read_niftis(subs)
    check_read_niftis(niftis, subs)

    # windows = [5, 10, 25, 45]
    window = 10

    clipped_niftis = clip_niftis(window, subs, timings, cues, niftis)
    check_clipped_niftis(clipped_niftis)

    return clipped_niftis

main()

c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting data from %s to %s" % (data.dtype.name, aux))
c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting data from %s to %s" % (data.dtype.name, aux))
c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting data from %s to %s" % (data.dtype.name, aux))
c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting data from %s to %s" % (data.dtype.name, aux))
c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\nilearn\image\resampling.py:545: UserWarning: Casting data from int16 to float32
  warnings.warn("Casting d

MemoryError: Unable to allocate 269. MiB for an array with shape (73, 92, 74, 142) and data type float32